# T7 It4: the decision rule around the judge (cut, extra queries, early exit) and the freeze

**Question:** It2 showed the candidate judge removes the fresh-NUDT wrong answers and gains on official under a hand-made cut (smallest zero-wrong
cut + one grid step). Here: which **cut rule** holds up on held-out sets, what happens on the **Jetson** replay (M = 5, extra queries one at a time),
and whether the judge can **stop early** (fewer LoFTR queries = less Jetson time). The rule is then **frozen** on the five seen sets
(`results/frozen_rules_t7.json`, `results/judge_t7.joblib`) before the blind NUDT 606 set is opened (It5).

**Protocol:** leave one set out for everything (judge and cut fitted on the other four sets). Each held-out judge is trained once
(`held_out_probabilities`) and reused by every replay. Gains: paired bootstrap against the agreement-free baselines (PC: bar 8 on the 6 standard
queries, else bar 11 on all 27; Jetson today: bar 8, second pass, cap 10, M = 5).

**Cut rules compared:**
- *zero-wrong + margin k*: the smallest cut with 0 wrong on the training sets at it and at every larger cut, then k grid steps up.
- *Learn-then-Test* (Angelopoulos et al. 2021; conformal risk control): walk the cuts from the strictest down while the one-sided 90 % Clopper-Pearson
  upper bound of the wrong rate among answered training photos stays <= alpha. With 0 wrong in n answers the bound is ~2.3 / n, so the rule also
  refuses cuts that are clean only on little data.

In [1]:
import os, sys, time, json
os.environ.setdefault("OMP_NUM_THREADS", str(len(os.sched_getaffinity(0))))   # one OpenMP thread per allowed core (more threads than cores slowed the judge ~100x)
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
import numpy as np, pandas as pd
from train.checks.candidate_judge import (FEATURES_CTX, FROZEN, FROZEN_VARIANTS, add_context, add_features, baseline, freeze, held_out_probabilities,
                                          leave_one_set_out, paired_gain, summary)
RES = ROOT / "tests/Test_Iteration_7/results"
SETS = ["official", "nudt101", "nudt202", "nudt404", "nudt505"]
table = pd.concat([pd.read_csv(RES / f"signals/{s}.csv.gz") for s in SETS], ignore_index=True)
table = add_context(add_features(table)).reset_index(drop=True)
t0 = time.time(); PROBS = held_out_probabilities(table, SETS, FEATURES_CTX); print("five held-out judges: %.0f s" % (time.time() - t0))
base_pc = baseline(table, 8, 11, min_choice=3)
base_jet = baseline(table, 8, 8, jetson=True, m=5)
display(pd.concat({"PC baseline": summary(base_pc), "Jetson today": summary(base_jet)})[["score", "wrong", "answered", "queries"]])

def run(name, base, **kw):
    s, rows = leave_one_set_out(table, SETS, FEATURES_CTX, probs=PROBS, min_choice=3, override=12, **kw)
    g = paired_gain(base, rows, SETS)
    s["cell"] = s.score.map("{:.2f}".format) + " / " + s.wrong.astype(str) + "w / t" + s.tau.astype(str) + " / " + s.held.map(g)
    return s.assign(rule=name), rows

five held-out judges: 11 s


score  wrong  answered  queries
             data                                      
PC baseline  official  49.963      0     0.850    9.491
             nudt101   54.840      0     0.917    8.450
             nudt202   57.060      0     0.960    7.470
             nudt404   54.240      2     0.940    7.575
             nudt505   54.240      1     0.917    8.380
Jetson today official  50.280      0     0.855    6.639
             nudt101   54.240      0     0.907    6.433
             nudt202   55.260      0     0.930    6.300
             nudt404   55.440      0     0.930    6.310
             nudt505   54.680      1     0.927    6.387

## 1. Cut rules on the PC (held-out score / wrong / cut / paired gain over the PC baseline)

In [2]:
out = []
for k in (0, 1, 2):
    out.append(run(f"zero-wrong + margin {k}", base_pc, margin_steps=k)[0])
for a in (0.002, 0.005, 0.01):
    out.append(run(f"Learn-then-Test alpha {a}", base_pc, cut_rule="ltt", alpha=a)[0])
cuts = pd.concat(out)
display(cuts.pivot(index="rule", columns="held", values="cell")[SETS])
cuts.to_csv(RES / "it4_cut_rules_pc.csv", index=False)

held,official,nudt101,nudt202,nudt404,nudt505
rule,,,,,
Learn-then-Test alpha 0.002,"51.01 / 0w / t0.99 / +1.04 [-0.22, +2.31]","55.32 / 0w / t0.99 / +0.48 [-0.12, +1.20]","56.46 / 0w / t0.99 / -0.60 [-1.50, +0.00]","56.22 / 0w / t0.99 / +1.98 [-0.06, +4.56]","55.76 / 0w / t0.99 / +1.52 [+0.52, +2.80]"
Learn-then-Test alpha 0.005,"55.16 / 0w / t0.9 / +5.19 [+3.58, +7.00]","55.72 / 0w / t0.95 / +0.88 [+0.08, +1.80]","57.06 / 0w / t0.925 / +0.00 [+0.00, +0.00]","56.70 / 0w / t0.9 / +2.46 [+0.48, +4.98]","56.96 / 0w / t0.9 / +2.72 [+1.36, +4.32]"
Learn-then-Test alpha 0.01,"56.36 / 0w / t0.7 / +6.40 [+4.69, +8.33]","54.56 / 3w / t0.6 / -0.28 [-2.28, +1.32]","55.26 / 3w / t0.6 / -1.80 [-5.10, +0.60]","56.70 / 0w / t0.8 / +2.46 [+0.48, +4.98]","56.36 / 1w / t0.75 / +2.12 [+0.52, +3.84]"
zero-wrong + margin 0,"54.52 / 0w / t0.925 / +4.56 [+3.04, +6.21]","55.72 / 0w / t0.95 / +0.88 [+0.08, +1.80]","56.46 / 0w / t0.95 / -0.60 [-1.50, +0.00]","56.70 / 0w / t0.95 / +2.46 [+0.48, +4.98]","56.36 / 0w / t0.95 / +2.12 [+0.92, +3.56]"
zero-wrong + margin 1,"54.36 / 0w / t0.95 / +4.40 [+2.88, +6.08]","55.72 / 0w / t0.97 / +0.88 [+0.08, +1.80]","56.46 / 0w / t0.97 / -0.60 [-1.50, +0.00]","56.70 / 0w / t0.97 / +2.46 [+0.48, +4.98]","55.96 / 0w / t0.97 / +1.72 [+0.60, +3.04]"
zero-wrong + margin 2,"53.00 / 0w / t0.97 / +3.04 [+1.77, +4.46]","55.52 / 0w / t0.98 / +0.68 [-0.04, +1.60]","56.46 / 0w / t0.98 / -0.60 [-1.50, +0.00]","56.52 / 0w / t0.98 / +2.28 [+0.30, +4.80]","55.96 / 0w / t0.98 / +1.72 [+0.60, +3.04]"


**Reading.** alpha 0.002 is too strict (cut 0.99); alpha 0.01 lets the cut fall to 0.6-0.8 and wrong answers come back (101, 202, 505).
alpha 0.005 is the best zero-wrong rule on the PC, but it sits close to that edge.

## 2. The same judge on the Jetson replay: a floor on the cut and a stricter cut for the extra queries

In [3]:
out, ROWS = [], {}
for name, kw in {"LTT 0.005": dict(), "LTT 0.005, floor 0.9": dict(tau_floor=0.9), "LTT 0.005, floor 0.925": dict(tau_floor=0.925),
                 "LTT 0.005, floor 0.9, extra-query cut 0.95": dict(tau_floor=0.9, wide_tau=0.95),
                 "LTT 0.005, floor 0.9, extra-query cut 0.97": dict(tau_floor=0.9, wide_tau=0.97)}.items():
    s, rows = run(name, base_jet, cut_rule="ltt", alpha=0.005, jetson=True, m=5, **kw)
    out.append(s); ROWS[name] = rows
jet = pd.concat(out)
display(jet.pivot(index="rule", columns="held", values="cell")[SETS])
r = ROWS["LTT 0.005, floor 0.9"]; print("wrong answers left with the floor alone:"); display(r[r.pts < 0])
jet.to_csv(RES / "it4_jetson_rules.csv", index=False)

held,official,nudt101,nudt202,nudt404,nudt505
rule,,,,,
LTT 0.005,"54.52 / 0w / t0.85 / +4.24 [+2.72, +5.89]","54.52 / 1w / t0.9 / +0.28 [-1.00, +1.48]","54.36 / 2w / t0.8 / -0.90 [-3.00, +0.60]","57.30 / 0w / t0.9 / +1.86 [+0.60, +3.30]","55.96 / 0w / t0.85 / +1.28 [+0.28, +2.48]"
"LTT 0.005, floor 0.9","53.48 / 0w / t0.9 / +3.20 [+1.77, +4.72]","54.52 / 1w / t0.9 / +0.28 [-1.00, +1.48]","55.26 / 0w / t0.9 / +0.00 [+0.00, +0.00]","57.30 / 0w / t0.9 / +1.86 [+0.60, +3.30]","55.76 / 0w / t0.9 / +1.08 [+0.00, +2.36]"
"LTT 0.005, floor 0.9, extra-query cut 0.95","53.16 / 0w / t0.9 / +2.88 [+1.52, +4.31]","54.92 / 0w / t0.9 / +0.68 [-0.24, +1.68]","55.26 / 0w / t0.9 / +0.00 [+0.00, +0.00]","57.00 / 0w / t0.9 / +1.56 [+0.48, +2.94]","55.56 / 0w / t0.9 / +0.88 [-0.12, +2.08]"
"LTT 0.005, floor 0.9, extra-query cut 0.97","53.16 / 0w / t0.9 / +2.88 [+1.52, +4.31]","54.72 / 0w / t0.9 / +0.48 [-0.36, +1.40]","55.26 / 0w / t0.9 / +0.00 [+0.00, +0.00]","57.00 / 0w / t0.9 / +1.56 [+0.48, +2.94]","55.56 / 0w / t0.9 / +0.88 [-0.12, +2.08]"
"LTT 0.005, floor 0.925","53.00 / 0w / t0.925 / +2.72 [+1.39, +4.08]","54.32 / 1w / t0.925 / +0.08 [-1.32, +1.28]","55.26 / 0w / t0.925 / +0.00 [+0.00, +0.00]","57.30 / 0w / t0.925 / +1.86 [+0.60, +3.30]","55.48 / 0w / t0.925 / +0.80 [-0.20, +2.00]"


wrong answers left with the floor alone:


,data,photo,stage,pts,err_sigma,inliers,p,n_queries,tau
436,nudt101,101,wide,-20,30.803613,4.0,0.928445,9,0.9


**Reading.** On the Jetson candidates (M = 5) Learn-then-Test picks lower cuts (0.8-0.85) and wrong answers appear. A floor of 0.9 (the lowest
cut any PC fold chose) removes two; the last one (NUDT 101 photo 101: a 4-inlier look-alike with P 0.93, found on the 9th query) is an
*extra-query* answer. More queries give a look-alike more chances, the same reason today's bars rise from 8 to 11 for the wider queries. A cut of
0.95 for the extra queries removes it, and at that setting no held-out set has a wrong answer.

## 3. Final rule on both replays, with and without the early exit

In [4]:
final, FINAL_ROWS = [], {}
for variant, kw in FROZEN_VARIANTS.items():
    base = base_jet if kw.get("jetson") else base_pc
    s, rows = run(variant, base, cut_rule="ltt", alpha=FROZEN["alpha"], tau_floor=FROZEN["tau_floor"], wide_tau=FROZEN["wide_tau"], **kw)
    final.append(s); FINAL_ROWS[variant] = rows
final = pd.concat(final)
display(final.pivot(index="rule", columns="held", values="cell").loc[list(FROZEN_VARIANTS)][SETS])
q = final.pivot(index="rule", columns="held", values="queries").loc[list(FROZEN_VARIANTS)][SETS]
print("mean LoFTR queries per photo:"); display(q)
final.to_csv(RES / "it4_final_rule_loso.csv", index=False)
pd.concat([r.assign(variant=v) for v, r in FINAL_ROWS.items()]).to_csv(RES / "it4_final_rule_loso_photos.csv", index=False)

held,official,nudt101,nudt202,nudt404,nudt505
rule,,,,,
pc,"54.84 / 0w / t0.9 / +4.88 [+3.26, +6.62]","55.72 / 0w / t0.9 / +0.88 [+0.08, +1.80]","57.06 / 0w / t0.9 / +0.00 [+0.00, +0.00]","56.70 / 0w / t0.9 / +2.46 [+0.48, +4.98]","56.76 / 0w / t0.9 / +2.52 [+1.20, +4.04]"
pc_early,"54.17 / 0w / t0.9 / +4.21 [+2.44, +6.09]","55.64 / 0w / t0.9 / +0.80 [-0.04, +1.80]","56.58 / 0w / t0.9 / -0.48 [-0.96, -0.12]","57.06 / 0w / t0.9 / +2.82 [+0.84, +5.40]","56.52 / 0w / t0.9 / +2.28 [+0.92, +3.84]"
jetson,"53.16 / 0w / t0.9 / +2.88 [+1.52, +4.31]","54.92 / 0w / t0.9 / +0.68 [-0.24, +1.68]","55.26 / 0w / t0.9 / +0.00 [+0.00, +0.00]","57.00 / 0w / t0.9 / +1.56 [+0.48, +2.94]","55.56 / 0w / t0.9 / +0.88 [-0.12, +2.08]"
jetson_early,"52.50 / 0w / t0.9 / +2.22 [+0.66, +3.82]","54.84 / 0w / t0.9 / +0.60 [-0.36, +1.60]","54.78 / 0w / t0.9 / -0.48 [-0.96, -0.12]","57.36 / 0w / t0.9 / +1.92 [+0.78, +3.30]","55.32 / 0w / t0.9 / +0.64 [-0.40, +1.88]"


mean LoFTR queries per photo:


held,official,nudt101,nudt202,nudt404,nudt505
rule,,,,,
pc,8.27,8.17,7.58,7.47,7.89
pc_early,4.37,3.86,3.16,2.97,3.46
jetson,6.40,6.39,6.30,6.22,6.32
jetson_early,2.51,2.08,1.89,1.72,1.95


## 4. Time on the Jetson

Model: 2.2 s per LoFTR query at M = 5 (measured in Test_Iteration_6) + the judge's signals (0.93 ms per candidate on one x86 core, i.e. ~5 x 0.93 ms
per query; taken x3 for the Jetson's slower CPU) + the judge itself (200 depth-3 trees, < 1 ms per query). The time classes (stated by the team):
**< 4 s good, 4-6 s viable, >= 6 s slow**, read here as the time a decision rule adds to (or removes from) a photo.

In [5]:
QUERY_S, SIGNAL_S = 2.2, 3 * 5 * 0.00093 + 0.001
jq = pd.concat({"Jetson today": summary(base_jet)["queries"],
                "jetson (judge)": final[final.rule == "jetson"].set_index("held")["queries"],
                "jetson_early (judge)": final[final.rule == "jetson_early"].set_index("held")["queries"]}, axis=1).loc[SETS]
sec = jq * QUERY_S
for col in ["jetson (judge)", "jetson_early (judge)"]:
    sec[col] += jq[col] * SIGNAL_S
sec.loc["mean"] = sec.mean()
display(sec.round(2).add_suffix(" [s / photo, matching + decision]"))
print("judge overhead per photo: %.0f ms (good)" % (1000 * jq['jetson (judge)'].mean() * SIGNAL_S))
print("early exit saves %.1f s per photo on average" % (sec.loc['mean', 'Jetson today'] - sec.loc['mean', 'jetson_early (judge)']))

,"Jetson today [s / photo, matching + decision]","jetson (judge) [s / photo, matching + decision]","jetson_early (judge) [s / photo, matching + decision]"
official,14.61,14.18,5.56
nudt101,14.15,14.15,4.61
nudt202,13.86,13.95,4.19
nudt404,13.88,13.78,3.81
nudt505,14.05,14.00,4.32
mean,14.11,14.01,4.50


judge overhead per photo: 95 ms (good)
early exit saves 9.6 s per photo on average


## 5. Freeze (fit on all five seen sets) before opening the blind set

In [6]:
import joblib
frozen = {"rule": {k: v for k, v in FROZEN.items()}, "variants": {}, "seen_sets": SETS,
          "note": "fixed before NUDT fresh 606 (seed 606, pool 2400, photos of fresh 404/505 excluded) was scored"}
for variant in FROZEN_VARIANTS:
    model, tau = freeze(table, SETS, variant)
    frozen["variants"][variant] = dict(tau=tau, **FROZEN_VARIANTS[variant])
    print(variant, "cut", tau)
joblib.dump(model, RES / "judge_t7.joblib")          # the model is the same for every variant (fitted on all labelled candidates)
json.dump(frozen, open(RES / "frozen_rules_t7.json", "w"), indent=1)
print(json.dumps(frozen, indent=1))

pc cut 0.9


pc_early cut 0.9


jetson cut 0.9


jetson_early cut 0.9
{
 "rule": {
  "features": [
   "li",
   "z",
   "rank",
   "coarse",
   "sharp",
   "hull",
   "cover9",
   "spread",
   "conf_inl",
   "hc_ratio",
   "scale_dev",
   "angle",
   "shift_px",
   "resid",
   "ln22",
   "ratio",
   "tfrac",
   "zgap",
   "ligap"
  ],
  "kind": "gbm",
  "alpha": 0.005,
  "tau_floor": 0.9,
  "wide_tau": 0.95,
  "override": 12,
  "min_choice": 3
 },
 "variants": {
  "pc": {
   "tau": 0.9
  },
  "pc_early": {
   "tau": 0.9,
   "early_tau": 0.98
  },
  "jetson": {
   "tau": 0.9,
   "jetson": true,
   "m": 5
  },
  "jetson_early": {
   "tau": 0.9,
   "jetson": true,
   "m": 5,
   "early_tau": 0.98
  }
 },
 "seen_sets": [
  "official",
  "nudt101",
  "nudt202",
  "nudt404",
  "nudt505"
 ],
 "note": "fixed before NUDT fresh 606 (seed 606, pool 2400, photos of fresh 404/505 excluded) was scored"
}


## Reading

- The judge with the frozen rule (Learn-then-Test alpha 0.005, floor 0.9, extra-query cut 0.95, override 12) has **0 wrong answers on every
  held-out set**, on the PC and on the Jetson replay.
- **Early exit** (answer after any standard query whose choice has P >= 0.98) cuts LoFTR queries from ~6.3 to ~2 per photo on the Jetson:
  about 9 s saved per photo, for a small score loss. It is the one rule here that changes the time class of the whole photo.
- What is left to show: the same numbers on photos no part of this rule has seen (It5, NUDT 606).